In [10]:
%%bash
FOLDSEEK_RESULTS="/home/ilnitsky/NPM/Arachnida_findings"
cut -f2 "$FOLDSEEK_RESULTS.txt" | cut -d'-' -f2 > "$FOLDSEEK_RESULTS.accessions.txt"

echo "$FOLDSEEK_RESULTS.accessions.txt"


/home/ilnitsky/NPM/Arachnida_findings.accessions.txt


In [1]:
import os
import requests
import subprocess
import pandas as pd
from typing import List, Dict, Tuple, Any
import re
import time
import xml.etree.ElementTree as ET
import json
from collections import defaultdict

# Define domain type to shape/color mapping based on Source_DB
DOMAIN_STYLES = {
    'pfam': ('RE', '#FF6347'),        # Tomato (highest priority)
    'cathgene3d': ('RE', '#4682B4'),  # Steel Blue
    'panther': ('RE', '#32CD32'),     # Lime Green
    'ssf': ('RE', '#FFD700'),         # Gold
    'profile': ('RE', '#DDA0DD'),     # Plum
    'pirsf': ('RE', '#FFA07A'),       # Light Salmon (lowest priority)
}

# Priority order for overlapping domains
PRIORITY = {
    'pfam': 5,
    'cathgene3d': 4,
    'panther': 3,
    'ssf': 2,
    'profile': 1,
    'pirsf': 0
}

# Default style for unknown source databases
DEFAULT_STYLE = ('RE', '#CCCCCC')  # Grey rectangle

def read_uniprot_ids(file_path: str) -> List[str]:
    """Read UniProt IDs from a file"""
    try:
        with open(file_path, 'r') as f:
            ids = [line.strip() for line in f if line.strip()]
        return ids
    except FileNotFoundError:
        print(f"Error: File {file_path} not found")
        return []
    except Exception as e:
        print(f"Error reading file: {e}")
        return []

def parse_fasta_header(header: str) -> Dict:
    """Parse UniProt FASTA header to extract taxonomy information"""
    organism_match = re.search(r'OS=(.*?)(?=\sOX=|\sPE=|\sGN=|\s\w+=|$)', header)
    taxonomy_match = re.search(r'OX=(\d+)', header)
    
    organism = organism_match.group(1) if organism_match else "Unknown"
    taxonomy_id = int(taxonomy_match.group(1)) if taxonomy_match else 0
    
    return {
        'organism': organism,
        'taxonomy_id': taxonomy_id
    }

def get_uniprot_data(uniprot_ids: List[str]) -> Dict[str, Dict]:
    """Fetch FASTA sequences and extract taxonomy information from headers"""
    base_url = "https://rest.uniprot.org/uniprotkb/"
    results = {}
    
    for uniprot_id in uniprot_ids:
        try:
            fasta_url = f"{base_url}{uniprot_id}.fasta"
            fasta_response = requests.get(fasta_url)
            
            if fasta_response.status_code == 200:
                fasta_text = fasta_response.text
                header = fasta_text.split('\n')[0]
                tax_info = parse_fasta_header(header)
                
                results[uniprot_id] = {
                    'fasta': fasta_text,
                    'taxonomy_id': tax_info['taxonomy_id'],
                    'organism': tax_info['organism']
                }
                print(f"Successfully processed {uniprot_id}")
            else:
                print(f"Failed to get FASTA for {uniprot_id}")
            
            time.sleep(0.1)
            
        except Exception as e:
            print(f"Error processing {uniprot_id}: {e}")
            continue
    
    return results

def sort_by_taxonomy(uniprot_data: Dict[str, Dict]) -> List[Dict]:
    """Sort the UniProt data by taxonomy ID and organism name"""
    sorted_data = sorted(
        uniprot_data.items(),
        key=lambda x: (x[1]['taxonomy_id'], x[1]['organism'])
    )
    return [{'id': k, **v} for k, v in sorted_data]

def get_taxonomy_lineage(taxid: int) -> Dict:
    """Get full taxonomic lineage from NCBI Taxonomy using E-utilities"""
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    efetch_url = f"{base_url}efetch.fcgi?db=taxonomy&id={taxid}&format=xml"
    
    try:
        response = requests.get(efetch_url)
        if response.status_code == 200:
            root = ET.fromstring(response.text)
            lineage_elem = root.find('.//Lineage')
            if lineage_elem is not None:
                lineage = lineage_elem.text.split('; ')
                if lineage[0] == "cellular organisms":
                    lineage = lineage[1:]
                formatted_lineage = '/'.join(lineage)
                return {
                    'taxid': taxid,
                    'lineage': formatted_lineage
                }
    except Exception as e:
        print(f"Error fetching taxonomy for {taxid}: {e}")
    
    return {'taxid': taxid, 'lineage': ''}

def get_secondary_structure_regions(uniprot_id: str, output_dir: str = "pdbs") -> List[Tuple[int, int, str]]:
    """Downloads AlphaFold PDB, runs alphafold_disorder.py, and returns beta strand and helix regions."""
    os.makedirs(output_dir, exist_ok=True)
    
    pdb_url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v4.pdb"
    pdb_path = os.path.join(output_dir, f"AF-{uniprot_id}-F1-model_v4.pdb")
    
    print(f"Downloading PDB for {uniprot_id}...")
    response = requests.get(pdb_url)
    if response.status_code == 200:
        with open(pdb_path, "w") as f:
            f.write(response.text)
    else:
        raise Exception(f"Failed to download PDB: HTTP {response.status_code}")
    
    disorder_script = "/home/ilnitsky/tools/AlphaFold-disorder/alphafold_disorder.py"
    output_tsv = os.path.join(output_dir, f"{uniprot_id}_af_disorder_data.tsv")
    
    print(f"Running alphafold_disorder.py for {uniprot_id}...")
    result = subprocess.run(
        [disorder_script, "-i", pdb_path, "-o", output_tsv],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise Exception(f"alphafold_disorder.py failed: {result.stderr}")
    
    print(f"Parsing {output_tsv}...")
    df = pd.read_csv(output_tsv, sep="\t")
    
    regions = []
    current_start = None
    current_type = None
    
    for index, row in df.iterrows():
        pos = row["pos"]
        ss = row["ss"]
        
        if ss in ["E", "H"]:
            if current_type is None:
                current_start = pos
                current_type = "Beta-strand" if ss == "E" else "Helix"
            elif ss != ("E" if current_type == "Beta-strand" else "H"):
                regions.append((current_start, pos - 1, current_type))
                current_start = pos
                current_type = "Beta-strand" if ss == "E" else "Helix"
        elif current_type is not None:
            regions.append((current_start, pos - 1, current_type))
            current_start = None
            current_type = None
    
    if current_type is not None:
        regions.append((current_start, df["pos"].iloc[-1], current_type))
    
    return regions

def get_interpro_domains(uniprot_id: str) -> List[Dict[str, Any]]:
    """Fetch domain information from InterPro for a given UniProt ID, including protein length."""
    databases = ["pfam", "panther", "cathgene3d", "ssf", "pirsf", "profile"]
    base_url = "https://www.ebi.ac.uk/interpro/api/entry"
    domains = []
    
    for db in databases:
        url = f"{base_url}/{db}/protein/uniprot/{uniprot_id}?extra_fields=description,go_terms"
        try:
            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                for result in data.get('results', []):
                    metadata = result.get('metadata', {})
                    proteins = result.get('proteins', [])
                    
                    if proteins and len(proteins) > 0:
                        protein = proteins[0]
                        protein_length = protein.get('protein_length', 0)
                        locations = protein.get('entry_protein_locations', [])
                        
                        if locations and len(locations) > 0:
                            location = locations[0]
                            fragments = location.get('fragments', [])
                            
                            if fragments and len(fragments) > 0:
                                fragment = fragments[0]
                                domain_info = {
                                    'source_database': metadata.get('source_database', ''),
                                    'accession': metadata.get('accession', ''),
                                    'name': metadata.get('name', ''),
                                    'start': fragment.get('start', 0),
                                    'end': fragment.get('end', 0),
                                    'score': location.get('score', None),
                                    'protein_length': protein_length
                                }
                                domains.append(domain_info)
            
            time.sleep(0.5)
            
        except Exception as e:
            print(f"Error fetching {db} domains for {uniprot_id}: {e}")
    
    return domains

def resolve_overlaps(domains: List[Tuple[int, int, str, str]]) -> List[Tuple[int, int, str, str]]:
    """Resolve overlapping domains based on priority."""
    if not domains:
        return []
    
    domains.sort(key=lambda x: (x[0], -PRIORITY.get(x[2], 0)))
    
    resolved = []
    current_end = -1
    
    for start, end, source_db, name in domains:
        if start > current_end:
            resolved.append((start, end, source_db, name))
            current_end = end
        elif PRIORITY.get(source_db, 0) > PRIORITY.get(resolved[-1][2], 0):
            resolved[-1] = (start, end, source_db, name)
            current_end = end
    
    return resolved

def format_itol_domains(proteins: Dict[str, Tuple[List[Tuple[int, int, str, str]], int]]) -> str:
    """Format domains into iTOL DATASET_DOMAINS format using actual protein length."""
    output = []
    
    output.append("DATASET_DOMAINS")
    output.append("SEPARATOR COMMA")
    output.append("DATASET_LABEL,Protein domains")
    output.append("COLOR,#000000")
    output.append("LABEL_AUTO_COLOR,1")
    output.append("BACKBONE_COLOR,#aaaaaa")
    output.append("BACKBONE_HEIGHT,10")
    output.append("BORDER_WIDTH,0")
    output.append("GRADIENT_FILL,0")
    output.append("DATA")
    
    for protein_id, (domains, protein_length) in proteins.items():
        resolved_domains = resolve_overlaps(domains)
        
        domain_str = []
        for start, end, source_db, name in resolved_domains:
            shape, color = DOMAIN_STYLES.get(source_db, DEFAULT_STYLE)
            domain_str.append(f"{shape}|{start}|{end}|{color}|{name}")
        
        if domain_str:
            protein_line = f"{protein_id},{protein_length},{','.join(domain_str)}"
            output.append(protein_line)
    
    return "\n".join(output)

def save_results(sorted_data: List[Dict], output_file: str = 'sorted_uniprot.fasta', species_only: bool = False):
    """Save the sorted FASTA sequences with lineage information, secondary structure regions,
    domain information, and iTOL domain dataset."""
    taxonomy_dict = {}
    ss_output_file = output_file.replace('.fasta', '_secondary_structure.txt')
    domain_output_file = output_file.replace('.fasta', '_domains.txt')
    itol_domain_file = output_file.replace('.fasta', '_domains_itol.txt')
    
    proteins_domains = defaultdict(lambda: ([], 0))  # (domains_list, protein_length)
    
    with open(output_file, 'w') as f_fasta, \
         open(ss_output_file, 'w') as f_ss, \
         open(domain_output_file, 'w') as f_domain:
        
        f_domain.write("Organism\tUniProt_ID\tSource_DB\tAccession\tName\tStart\tEnd\tScore\n")
        
        for i, entry in enumerate(sorted_data):
            tax_info = get_taxonomy_lineage(entry['taxonomy_id'])
            taxonomy_dict[entry['id']] = {
                'taxid': tax_info['taxid'],
                'lineage': tax_info['lineage'],
                'organism': entry['organism']
            }
            
            fasta_lines = entry['fasta'].split('\n')
            header = fasta_lines[0]
            sequence = '\n'.join(line for line in fasta_lines[1:] if line.strip())
            
            organism_name = entry['organism'].replace(' ', '_')
            uniprot_id = entry['id']
            combined_id = f"{organism_name}-{uniprot_id}"
            
            try:
                if header.startswith('>'):
                    header = header[1:]
                
                id_part = header.split(' ')[0]
                
                if species_only:
                    new_header = f">{combined_id}"
                else:
                    desc_part = ' '.join(header.split(' ')[1:])
                    new_header = f">{id_part} {desc_part.split(' OS=')[0]} {tax_info['lineage']}/ {entry['organism']} OX={entry['taxonomy_id']}"
                    if 'GN=' in desc_part:
                        new_header += f" {' '.join(part for part in desc_part.split(' ') if part.startswith(('GN=', 'PE=', 'SV=')))}"
                
            except Exception as e:
                print(f"Error formatting header for entry {entry.get('id', 'unknown')}: {e}")
                new_header = f">{combined_id}" if species_only else f"{header} {tax_info['lineage']}/"
            
            f_fasta.write(f"{new_header}\n")
            f_fasta.write(f"{sequence}\n")

            try:
                regions = get_secondary_structure_regions(uniprot_id)
                for start, end, struct_type in regions:
                    f_ss.write(f"{combined_id}\t{start}\t{end}\t{struct_type}\n")
            except Exception as e:
                print(f"Error processing secondary structure for {uniprot_id}: {e}")
                
            try:
                print(f"Fetching domain information for {uniprot_id}...")
                domains = get_interpro_domains(uniprot_id)
                for domain in domains:
                    f_domain.write(f"{organism_name}\t{uniprot_id}\t{domain['source_database']}\t{domain['accession']}\t{domain['name']}\t{domain['start']}\t{domain['end']}\t{domain['score'] if domain['score'] is not None else 'null'}\n")
                    domain_list, _ = proteins_domains[combined_id]
                    domain_list.append((domain['start'], domain['end'], domain['source_database'], domain['name']))
                    proteins_domains[combined_id] = (domain_list, domain['protein_length'])
            except Exception as e:
                print(f"Error processing domain information for {uniprot_id}: {e}")

    # Generate iTOL domain dataset
    with open(itol_domain_file, 'w') as f_itol:
        itol_output = format_itol_domains(proteins_domains)
        f_itol.write(itol_output)
        print(f"iTOL domain dataset written to {itol_domain_file}")

    # Create Newick tree file with paralogues grouped
    with open(output_file.replace('.fasta', '.nwk'), 'w') as f:
        tree = {}
        # Group UniProt IDs by organism first
        organisms = defaultdict(list)
        for entry_id, tax_data in taxonomy_dict.items():
            organism = tax_data['organism'].replace(' ', '_')
            organisms[organism].append(entry_id)
        
        # Build the taxonomic tree
        for organism, uniprot_ids in organisms.items():
            lineage = taxonomy_dict[uniprot_ids[0]]['lineage'].split('/')
            current_node = tree
            
            for level in lineage:
                if level:
                    level = level.strip().replace(' ', '_')
                    if level not in current_node:
                        current_node[level] = {}
                    current_node = current_node[level]
            
            # Add organism node with grouped UniProt IDs
            if len(uniprot_ids) > 1:
                paralogue_str = ','.join(f"{organism}-{uid}" for uid in uniprot_ids)
                current_node[organism] = {f"({paralogue_str}){organism}": {}}
            else:
                current_node[organism] = {f"{organism}-{uniprot_ids[0]}": {}}

        def dict_to_newick(node):
            if not node:
                return ""
            children = []
            for key, value in node.items():
                child_str = dict_to_newick(value)
                if child_str:
                    children.append(f"({child_str}){key}")
                else:
                    children.append(key)
            return ','.join(children)

        newick_str = dict_to_newick(tree)
        f.write(f"({newick_str});")

def main():
    input_file = "/home/ilnitsky/NPM/Arachnida_findings.accessions.txt"
    uniprot_ids = read_uniprot_ids(input_file)
    
    if not uniprot_ids:
        print("No UniProt IDs found in the input file")
        return
    
    print(f"Found {len(uniprot_ids)} UniProt IDs to process")
    print("Fetching UniProt data...")
    uniprot_data = get_uniprot_data(uniprot_ids)
    
    if not uniprot_data:
        print("No data was retrieved from UniProt")
        return
        
    print("Sorting by taxonomy...")
    sorted_data = sort_by_taxonomy(uniprot_data)
    
    print("Saving results...")
    save_results(sorted_data, species_only=True)
    
    print("Done! Results saved to sorted_uniprot.fasta, sorted_uniprot_secondary_structure.txt, sorted_uniprot_domains.txt, and sorted_uniprot_domains_itol.txt")

if __name__ == "__main__":
    main()

Found 38 UniProt IDs to process
Fetching UniProt data...
Successfully processed A0A293MDK7
Successfully processed W4VS53
Successfully processed A0A147BAD8
Successfully processed A0A293MDK7
Successfully processed A0A834RGE5
Successfully processed A0A4Y2KZB4
Successfully processed A0A3G5AP41
Successfully processed A0A2L2YUH5
Successfully processed A0A4Y2QJK7
Successfully processed A0A293MDK7
Successfully processed A0A1Z5KYA8
Successfully processed A0A6G1SAJ6
Successfully processed A0A147BAD8
Successfully processed A0A7R9PT64
Successfully processed A0A7R9L173
Successfully processed A0A4Y2KZB4
Successfully processed A0A3G5AP41
Successfully processed B7Q9T4
Successfully processed A0A132ALI0
Successfully processed A0A1V9X6D5
Successfully processed A0A834RGE5
Successfully processed A0A443RRW4
Successfully processed A0A443SEY6
Successfully processed A0A6M2E5Y6
Successfully processed A0A6P6YJT1
Successfully processed A0A4Y2UNY9
Successfully processed A0A4Y2EWY1
Successfully processed T1L6A2
Suc

In [ ]:
!conda activate mobidb
!mobidb_lite sorted_uniprot.fasta mobidb_sorted_uniprot.txt




In [2]:
!cat mobidb_sorted_uniprot.txt sorted_uniprot_secondary_structure.txt | sort > full_prot_annotation.txt

In [3]:
from collections import defaultdict
import re

# Define domain type to shape/color mapping
DOMAIN_STYLES = {
    'Low complexity': ('RE', '#FFB6C1'),  # Light pink
    'Polyampholyte': ('HH', '#87CEEB'),   # Sky blue
    'Polar': ('EL', '#98FB98'),           # Pale green
    'Neg': ('DI', '#FFA07A'),  # Light salmon
    'Pos': ('TR', '#DDA0DD'),  # Plum
    'Proline-rich': ('OC', '#F0E68C'),    # Khaki
    'Glycine-rich': ('HV', '#E6E6FA'),    # Lavender
    'Beta': ('RE', '#4682B4'),     # Steel blue
    'Helix': ('RE', '#FF8C00'),           # Dark orange
}

# Default style for unknown domain types
DEFAULT_STYLE = ('RE', '#CCCCCC')  # Grey rectangle

def parse_mobidb(file_path):
    proteins = defaultdict(list)
    current_protein = None
    
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 4:
                protein_id = parts[0]
                start = int(parts[1])
                end = int(parts[2])
                domain_type = parts[3]
                
                # Only store entries with actual domain types (not just '-')
                if domain_type != '-':
                    proteins[protein_id].append((start, end, domain_type))
    
    return proteins

def format_itol_domains(proteins):
    output = []
    
    # Add header
    output.append("DATASET_DOMAINS")
    output.append("SEPARATOR COMMA")
    output.append("DATASET_LABEL,Protein domains")
    output.append("COLOR,#000000")
    output.append("LABEL_AUTO_COLOR,1")
    output.append("BACKBONE_COLOR,#aaaaaa")
    output.append("BACKBONE_HEIGHT,10")
    output.append("BORDER_WIDTH,0")
    output.append("GRADIENT_FILL,0")
    output.append("DATA")
    
    # Process each protein
    for protein_id, domains in proteins.items():
        # Find the maximum end position to determine protein length
        protein_length = max(end for _, end, _ in domains)
        
        # Format domains
        domain_str = []
        for start, end, domain_type in domains:
            shape, color = DOMAIN_STYLES.get(domain_type, DEFAULT_STYLE)
            domain_str.append(f"{shape}|{start}|{end}|{color}|{domain_type}")
        
        # Combine into final string
        protein_line = f"{protein_id},{protein_length},{','.join(domain_str)}"
        output.append(protein_line)
    
    return "\n".join(output)

# Main execution
proteins = parse_mobidb("full_prot_annotation.txt")
itol_output = format_itol_domains(proteins)

# Write output to file
with open("protein_domains.txt", "w") as f:
    f.write(itol_output)

In [2]:
from collections import defaultdict

# Define domain type to shape/color mapping based on Source_DB
DOMAIN_STYLES = {
    'pfam': ('RE', '#FF6347'),        # Tomato (highest priority)
    'cathgene3d': ('RE', '#4682B4'),  # Steel Blue
    'panther': ('RE', '#32CD32'),     # Lime Green
    'ssf': ('RE', '#FFD700'),         # Gold
    'profile': ('RE', '#DDA0DD'),     # Plum
    'pirsf': ('RE', '#FFA07A'),       # Light Salmon (lowest priority)
}

# Priority order for overlapping domains
PRIORITY = {
    'pfam': 5,
    'cathgene3d': 4,
    'panther': 3,
    'ssf': 2,
    'profile': 1,
    'pirsf': 0
}

# Default style for unknown source databases
DEFAULT_STYLE = ('RE', '#CCCCCC')  # Grey rectangle

def parse_uniprot_domains(file_path):
    proteins = defaultdict(list)
    
    with open(file_path, 'r') as f:
        next(f)  # Skip header
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 7:
                protein_id = parts[1]  # UniProt_ID
                source_db = parts[2]   # Source_DB
                name = parts[4]        # Name (instead of Accession)
                start = int(parts[5])  # Start
                end = int(parts[6])    # End
                
                # Store domain entry with source_db and name
                proteins[protein_id].append((start, end, source_db, name))
    
    return proteins

def resolve_overlaps(domains):
    """Resolve overlapping domains based on priority."""
    if not domains:
        return []
    
    # Sort by start position, then by priority (descending)
    domains.sort(key=lambda x: (x[0], -PRIORITY.get(x[2], 0)))
    
    resolved = []
    current_end = -1
    
    for start, end, source_db, name in domains:
        # If no overlap with previous, add directly
        if start > current_end:
            resolved.append((start, end, source_db, name))
            current_end = end
        # If overlap, only add if higher priority (already sorted by priority)
        elif PRIORITY.get(source_db, 0) > PRIORITY.get(resolved[-1][2], 0):
            resolved[-1] = (start, end, source_db, name)
            current_end = end
    
    return resolved

def format_itol_domains(proteins):
    output = []
    
    # Add iTOL header
    output.append("DATASET_DOMAINS")
    output.append("SEPARATOR COMMA")
    output.append("DATASET_LABEL,Protein domains")
    output.append("COLOR,#000000")
    output.append("LABEL_AUTO_COLOR,1")
    output.append("BACKBONE_COLOR,#aaaaaa")
    output.append("BACKBONE_HEIGHT,10")
    output.append("BORDER_WIDTH,0")
    output.append("GRADIENT_FILL,0")
    output.append("DATA")
    
    # Process each protein
    for protein_id, domains in proteins.items():
        # Resolve overlapping domains
        resolved_domains = resolve_overlaps(domains)
        
        # Find the maximum end position to determine protein length
        protein_length = max(end for _, end, _, _ in resolved_domains) if resolved_domains else 0
        
        # Format domains
        domain_str = []
        for start, end, source_db, name in resolved_domains:
            shape, color = DOMAIN_STYLES.get(source_db, DEFAULT_STYLE)
            domain_str.append(f"{shape}|{start}|{end}|{color}|{name}")
        
        # Combine into final string
        if domain_str:  # Only include proteins with domains
            protein_line = f"{protein_id},{protein_length},{','.join(domain_str)}"
            output.append(protein_line)
    
    return "\n".join(output)

# Main execution
file_path = "sorted_uniprot_domains.txt"
proteins = parse_uniprot_domains(file_path)
itol_output = format_itol_domains(proteins)

# Write output to file
with open("uniprot_domains_itol.txt", "w") as f:
    f.write(itol_output)

print("iTOL domain file written to 'uniprot_domains_itol.txt'")

iTOL domain file written to 'uniprot_domains_itol.txt'
